In [ ]:
# SHRUG CELL 0 — setup, shared helpers, unzip, inventory
import pandas as pd, numpy as np, os, glob, zipfile
from google.colab import drive; drive.mount('/content/drive')

BASE = '/content/drive/My Drive/Data/Elevar'
SRC  = f'{BASE}/shrug_zips'      # put the downloaded .zip files here
DST  = f'{BASE}/shrug'           # extracted CSVs land here
os.makedirs(SRC, exist_ok=True); os.makedirs(DST, exist_ok=True)

KEYS = ['Dist Code','Year','State Code','State Name','Dist Name']
def rd(f):                        # ICRISAT reader — sentinel + year handling
    p = os.path.join(BASE, f)
    try: d = pd.read_csv(p, low_memory=False)
    except UnicodeDecodeError: d = pd.read_csv(p, encoding='latin-1', low_memory=False)
    if 'Year' in d.columns:
        d['Year'] = pd.to_numeric(d['Year'].astype(str).str.extract(r'(\d{4})')[0], errors='coerce')
    v = [c for c in d.columns if c not in KEYS]
    d[v] = d[v].apply(pd.to_numeric, errors='coerce').mask(lambda x: x < 0)
    return d[d['Dist Code'] < 800]

def dm(df, cols, g1='Dist Code', g2='Year', it=12):   # two-way demeaning
    X = df[cols].astype(float).copy()
    for _ in range(it):
        X = X - X.groupby(df[g1].values).transform('mean')
        X = X - X.groupby(df[g2].values).transform('mean')
    return X

# ---- unzip everything and report what's inside ----
for z in sorted(glob.glob(f'{SRC}/*.zip')):
    with zipfile.ZipFile(z) as f:
        print(f'### {os.path.basename(z)}')
        for n in f.namelist(): print('   ', n)
        f.extractall(DST)
        print()

# ---- inventory the extracted CSVs: shape + columns, no full loads ----
for p in sorted(glob.glob(f'{DST}/**/*.csv', recursive=True)):
    try:
        head = pd.read_csv(p, nrows=3, low_memory=False)
        n = sum(1 for _ in open(p, encoding='utf-8', errors='ignore')) - 1
        print(f'### {os.path.relpath(p, DST)}  rows≈{n:,}  cols={len(head.columns)}')
        print('   ', ' | '.join(head.columns[:40]))
        if len(head.columns) > 40: print(f'    ... +{len(head.columns)-40} more')
        print()
    except Exception as e:
        print(f'!! {os.path.relpath(p, DST)}: {e}\n')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
### SHRUG GP to shrids.zip
    Shrid To Local Governance Body Matching/
    Shrid To Local Governance Body Matching/.DS_Store
    Shrid To Local Governance Body Matching/citation.txt
    Shrid To Local Governance Body Matching/code/
    Shrid To Local Governance Body Matching/code/.Rproj.user/
    Shrid To Local Governance Body Matching/code/.Rproj.user/E85E44BD/
    Shrid To Local Governance Body Matching/code/.Rproj.user/E85E44BD/bibliography-index/
    Shrid To Local Governance Body Matching/code/.Rproj.user/E85E44BD/ctx/
    Shrid To Local Governance Body Matching/code/.Rproj.user/E85E44BD/explorer-cache/
    Shrid To Local Governance Body Matching/code/.Rproj.user/E85E44BD/pcs/
    Shrid To Local Governance Body Matching/code/.Rproj.user/E85E44BD/pcs/files-pane.pper
    Shrid To Local Governance Body Matching/code/.Rproj.user/E85E44BD/pcs/source-pane.ppe

In [ ]:
# SHRUG CELL 1 — load the district-level tables only
import pandas as pd, glob, os
DST = '/content/drive/My Drive/Data/Elevar/shrug'
def find(name):
    hits = glob.glob(f'{DST}/**/{name}', recursive=True)
    return hits[0] if hits else None

WANT = {
 'dmsp'      : 'dmsp_pc11dist.csv',
 'viirs'     : 'viirs_annual_pc11dist.csv',
 'pca11'     : 'pc11_pca_clean_pc11dist.csv',
 'pca01'     : 'pc01_pca_clean_pc01dist.csv',
 'pca91'     : 'pc91_pca_clean_pc91dist.csv',
 'vd11'      : 'pc11_vd_clean_pc11dist.csv',
 'vd01'      : 'pc01_vd_clean_pc01dist.csv',
 'vd91'      : 'pc91_vd_clean_pc91dist.csv',
 'td11'      : 'pc11_td_clean_pc11dist.csv',
 'ec13'      : 'ec13_pc11dist.csv',
 'ec05'      : 'ec05_pc01dist.csv',
 'ec90'      : 'ec90_pc91dist.csv',
 'secc_cons' : 'secc_cons_rural_pc11dist.csv',
 'secc_rural': 'secc_rural_pc11dist.csv',
 'elev'      : 'elevation_pc11dist.csv',
 'tri'       : 'tri_pc11dist.csv',
 'vcf'       : 'vcf_pc11dist.csv',
 'pm25'      : 'pm25_pc11dist.csv',
 'distwt'    : 'dist_pc11_pop_area_key.csv',
 'names'     : 'shrid_loc_names.csv',
}
S = {}
for k, fn in WANT.items():
    p = find(fn)
    if p is None: print(f'!! missing {fn}'); continue
    S[k] = pd.read_csv(p, low_memory=False)
    print(f'{k:11s} {S[k].shape}')

# district name key: 640 rows, needed for the ICRISAT bridge
nm = S['names'][['state_name','district_name']].drop_duplicates()
key = (pd.read_csv(find('shrid_pc11dist_key.csv'))
        .merge(S['names'][['shrid2','state_name','district_name']], on='shrid2')
        [['pc11_state_id','pc11_district_id','state_name','district_name']]
        .drop_duplicates(['pc11_state_id','pc11_district_id']))
key.to_csv('/content/pc11_district_names.csv', index=False)
print('\npc11 districts:', len(key))
print(key.head().to_string(index=False))

# sanity: is the calibrated light series the one to use?
d = S['dmsp']
print('\nDMSP years:', d.year.min(), '-', d.year.max(), '| districts:', d.pc11_district_id.nunique())
print(d[['dmsp_total_light','dmsp_total_light_cal']].describe().round(1).to_string())
print('\nVIIRS years:', S['viirs'].year.min(), '-', S['viirs'].year.max())

dmsp        (20480, 10)
viirs       (15360, 9)
pca11       (640, 87)
pca01       (593, 57)
pca91       (452, 59)
vd11        (631, 281)
vd01        (581, 107)
vd91        (445, 97)
td11        (635, 313)
ec13        (640, 140)
ec05        (591, 138)
ec90        (425, 129)
secc_cons   (615, 10)
secc_rural  (615, 104)
elev        (640, 10)
tri         (640, 8)
vcf         (12800, 7)
pm25        (14720, 7)
distwt      (640, 4)
names       (596389, 7)

pc11 districts: 640
 pc11_state_id  pc11_district_id    state_name district_name
             1                 1 jammu kashmir       kupwara
             1                 2 jammu kashmir        badgam
             1                 3 jammu kashmir    leh ladakh
             1                 4 jammu kashmir        kargil
             1                 5 jammu kashmir         punch

DMSP years: 1994 - 2013 | districts: 640
       dmsp_total_light  dmsp_total_light_cal
count           20480.0               20480.0
mean            21774.7    

In [ ]:
# SHRUG CELL 2 — is night light a usable demand proxy, and does it respond to rainfall at all?
import pandas as pd, numpy as np, statsmodels.api as sm

d = S['dmsp'].copy()
d = d[d.dmsp_total_light_cal > 0]
d['ln_light'] = np.log(d.dmsp_total_light_cal)
d['dist_id']  = d.pc11_state_id*1000 + d.pc11_district_id

# --- 1. does it behave like an economic series? ---
yr = d.groupby('year').ln_light.mean()
print('mean ln(light) by year:'); print(yr.round(3).to_string())
print(f'\n1994->2013 growth: {np.exp(yr.iloc[-1]-yr.iloc[0])-1:+.1%}  (expect strongly positive)')

# --- 2. cross-section validation against SECC consumption (2012) ---
c = S['secc_cons'][['pc11_state_id','pc11_district_id','secc_cons_pc_rural','secc_pov_rate_rural']]
w = S['distwt']
v = (d[d.year.between(2011,2013)].groupby(['pc11_state_id','pc11_district_id'])
       .dmsp_total_light_cal.mean().rename('light').reset_index()
       .merge(c, on=['pc11_state_id','pc11_district_id'])
       .merge(w, on=['pc11_state_id','pc11_district_id']))
v['light_pc'] = v.light/v.dist_pc11_pca_tot_p
v = v.replace([np.inf,-np.inf], np.nan).dropna()
print(f'\nVALIDATION n={len(v)}')
print(f'  corr( ln light per capita , SECC consumption pc )  = {np.log(v.light_pc).corr(v.secc_cons_pc_rural):+.3f}')
print(f'  corr( ln light per capita , rural poverty rate  )  = {np.log(v.light_pc).corr(v.secc_pov_rate_rural):+.3f}')
print('  want: consumption POSITIVE, poverty NEGATIVE, both |r| > 0.4')

# --- 3. does light respond to rainfall? crude first look, ICRISAT districts only ---
# (proper version waits for the bridge; here we just check the signal exists)
print(f'\npanel: {d.dist_id.nunique()} districts x {d.year.nunique()} years = {len(d)} obs')
print('within-district SD of ln_light:', round(d.groupby("dist_id").ln_light.std().mean(),3))
print('  (needs to be well above 0 or there is no year-to-year variation to explain)')

mean ln(light) by year:
year
1994     9.042
1995     9.284
1996     9.316
1997     9.178
1998     9.344
1999     9.251
2000     9.698
2001     9.401
2002     9.606
2003     9.556
2004     9.621
2005     9.493
2006     9.559
2007     9.857
2008     9.741
2009     9.857
2010    10.208
2011    10.136
2012    10.130
2013    10.140

1994->2013 growth: +199.8%  (expect strongly positive)

VALIDATION n=615
  corr( ln light per capita , SECC consumption pc )  = +0.318
  corr( ln light per capita , rural poverty rate  )  = -0.246
  want: consumption POSITIVE, poverty NEGATIVE, both |r| > 0.4

panel: 640 districts x 20 years = 20350 obs
within-district SD of ln_light: 0.418
  (needs to be well above 0 or there is no year-to-year variation to explain)


In [ ]:
# SHRUG CELL 3 — pick the light specification that actually tracks rural welfare
import pandas as pd, numpy as np, statsmodels.api as sm

d = S['dmsp'].copy(); d = d[d.dmsp_total_light_cal > 0]
L = (d[d.year.between(2011,2013)].groupby(['pc11_state_id','pc11_district_id'])
       [['dmsp_total_light_cal','dmsp_mean_light_cal','dmsp_num_cells']].mean().reset_index())

w  = S['distwt']
rp = S['vd11'][['pc11_state_id','pc11_district_id','pc11_vd_t_p']].rename(columns={'pc11_vd_t_p':'rural_pop'})
c  = S['secc_cons'][['pc11_state_id','pc11_district_id','secc_cons_pc_rural','secc_pov_rate_rural']]

V = (L.merge(w, on=['pc11_state_id','pc11_district_id'])
       .merge(rp, on=['pc11_state_id','pc11_district_id'], how='left')
       .merge(c,  on=['pc11_state_id','pc11_district_id']))

V['A_tot_pc']    = V.dmsp_total_light_cal / V.dist_pc11_pca_tot_p
V['B_tot_rural'] = V.dmsp_total_light_cal / V.rural_pop
V['C_mean_cell'] = V.dmsp_mean_light_cal
V['D_per_area']  = V.dmsp_total_light_cal / V.dist_pc11_land_area
V['E_urbanfree'] = V.dmsp_total_light_cal / V.dmsp_num_cells   # density, not level
V = V.replace([np.inf,-np.inf], np.nan)

print('CORRELATION WITH RURAL WELFARE  (n varies by missingness)')
print(f"{'spec':14s} {'n':>5s} {'r_cons':>8s} {'r_pov':>8s} {'r_cons|state FE':>17s}")
for k, lab in [('A_tot_pc','tot/pop'), ('B_tot_rural','tot/rural pop'),
               ('C_mean_cell','mean per cell'), ('D_per_area','tot/area'), ('E_urbanfree','tot/cells')]:
    v = V[[k,'secc_cons_pc_rural','secc_pov_rate_rural','pc11_state_id']].dropna()
    v = v[v[k] > 0]; x = np.log(v[k])
    rc, rp_ = x.corr(v.secc_cons_pc_rural), x.corr(v.secc_pov_rate_rural)
    # within-state: demean both by state, which is what a district score compares
    xs = x - x.groupby(v.pc11_state_id).transform('mean')
    ys = v.secc_cons_pc_rural - v.secc_cons_pc_rural.groupby(v.pc11_state_id).transform('mean')
    print(f'{lab:14s} {len(v):5d} {rc:+8.3f} {rp_:+8.3f} {xs.corr(ys):+17.3f}')

# how urban-dominated is the light? drop the brightest decile and retest the base spec
v = V[['A_tot_pc','secc_cons_pc_rural']].dropna(); v = v[v.A_tot_pc>0]
cut = v.A_tot_pc.quantile(0.90)
sub = v[v.A_tot_pc < cut]
print(f'\ndropping brightest decile (n={len(sub)}): r_cons = '
      f'{np.log(sub.A_tot_pc).corr(sub.secc_cons_pc_rural):+.3f}  (was {np.log(v.A_tot_pc).corr(v.secc_cons_pc_rural):+.3f})')
print('  if this JUMPS, metros were dragging the fit and the rural signal is stronger than it looked')

CORRELATION WITH RURAL WELFARE  (n varies by missingness)
spec               n   r_cons    r_pov   r_cons|state FE
tot/pop          615   +0.318   -0.246            +0.126
tot/rural pop    615   +0.517   -0.412            +0.276
mean per cell    615   +0.637   -0.618            +0.461
tot/area         615   +0.639   -0.608            +0.388
tot/cells        615   +0.637   -0.618            +0.461

dropping brightest decile (n=553): r_cons = +0.393  (was +0.318)
  if this JUMPS, metros were dragging the fit and the rural signal is stronger than it looked


In [ ]:
# SHRUG CELL 4 — H4 dry run: does light respond to rainfall, and does non-farm density damp it?
import pandas as pd, numpy as np, statsmodels.api as sm

d = S['dmsp'].copy(); d = d[d.dmsp_mean_light_cal > 0]
d['ln_light'] = np.log(d.dmsp_mean_light_cal)
d['did'] = d.pc11_state_id*1000 + d.pc11_district_id

# non-farm density, pc11 2011: household industry + other workers, share of total workers
p = S['pca11']
p['nonfarm_sh'] = (p.pc11_pca_main_hh_p + p.pc11_pca_main_ot_p) / p.pc11_pca_tot_work_p
p['hhind_sh']   =  p.pc11_pca_main_hh_p / p.pc11_pca_tot_work_p     # the entrepreneurial household
p['did'] = p.pc11_state_id*1000 + p.pc11_district_id
print('nonfarm_sh:', p.nonfarm_sh.describe().round(3).to_dict())
print('hhind_sh  :', p.hhind_sh.describe().round(3).to_dict())

# EC13 non-farm employment per capita — the firm-side measure
e = S['ec13'][['pc11_state_id','pc11_district_id','ec13_emp_all']].copy()
e['did'] = e.pc11_state_id*1000 + e.pc11_district_id
e = e.merge(S['distwt'], on=['pc11_state_id','pc11_district_id'])
e['ec_emp_pc'] = e.ec13_emp_all / e.dist_pc11_pca_tot_p

M = d.merge(p[['did','nonfarm_sh','hhind_sh']], on='did').merge(e[['did','ec_emp_pc']], on='did')
print(f'\npanel {M.did.nunique()} districts x {M.year.nunique()} yrs = {len(M)}')
print('corr(nonfarm_sh, ec_emp_pc) =', round(M.drop_duplicates("did").nonfarm_sh.corr(M.drop_duplicates("did").ec_emp_pc),3),
      ' <- two independent non-farm measures; want decently positive')

def dm(df, cols, it=12):
    X = df[cols].astype(float).copy()
    for _ in range(it):
        X = X - X.groupby(df['did'].values).transform('mean')
        X = X - X.groupby(df['year'].values).transform('mean')
    return X

# placeholder shock: district's own ln_light deviation is NOT a shock — we need rainfall.
# until the bridge lands, just establish the FE machinery works and report within-variation.
D = dm(M, ['ln_light'])
print('\nresidual SD of ln_light after two-way FE:', round(D.ln_light.std(),4))
print('  this is the variation H4 will try to explain with rainfall. >0.1 is workable.')

M.drop_duplicates('did')[['did','nonfarm_sh','hhind_sh','ec_emp_pc']].to_parquet(
    '/content/pc11_nonfarm.parquet', index=False)
print('saved pc11_nonfarm.parquet')

nonfarm_sh: {'count': 640.0, 'mean': 0.351, 'std': 0.194, 'min': 0.081, '25%': 0.205, '50%': 0.298, '75%': 0.444, 'max': 0.953}
hhind_sh  : {'count': 640.0, 'mean': 0.022, 'std': 0.018, 'min': 0.002, '25%': 0.012, '50%': 0.017, '75%': 0.027, 'max': 0.168}

panel 640 districts x 20 yrs = 20350
corr(nonfarm_sh, ec_emp_pc) = 0.507  <- two independent non-farm measures; want decently positive

residual SD of ln_light after two-way FE: 0.451
  this is the variation H4 will try to explain with rainfall. >0.1 is workable.
saved pc11_nonfarm.parquet


In [ ]:
import shutil
shutil.copy('/content/pc11_district_names.csv',
            '/content/drive/My Drive/Data/Elevar/pc11_district_names.csv')

'/content/drive/My Drive/Data/Elevar/pc11_district_names.csv'

In [ ]:
# SHRUG CELL 5 — H4: does non-farm density buffer local economic activity against monsoon failure?
import pandas as pd, numpy as np, statsmodels.api as sm
BASE = '/content/drive/My Drive/Data/Elevar'

BR = pd.read_csv(f'{BASE}/pc11_to_icrisat_bridge.csv')
W  = S['distwt'].rename(columns={'dist_pc11_pca_tot_p':'pop'})

# ---- lights -> ICRISAT parents, population-weighted ----
d = S['dmsp'].copy(); d = d[d.dmsp_mean_light_cal > 0]
d = d.merge(BR[['pc11_state_id','pc11_district_id','dist_code']], on=['pc11_state_id','pc11_district_id'])
d = d.merge(W[['pc11_state_id','pc11_district_id','pop']], on=['pc11_state_id','pc11_district_id'])
d['wl'] = d.dmsp_mean_light_cal * d['pop']
L = (d.groupby(['dist_code','year'])
       .apply(lambda g: pd.Series({'light': g.wl.sum()/g['pop'].sum()}))
       .reset_index())
L['ln_light'] = np.log(L.light)
print('light panel:', L.shape, L.year.min(), '-', L.year.max(), '|', L.dist_code.nunique(), 'districts')

# ---- non-farm density -> parents, population-weighted ----
nf = pd.read_parquet('/content/pc11_nonfarm.parquet')
nf['pc11_state_id']    = nf.did // 1000
nf['pc11_district_id'] = nf.did %  1000
nf = (nf.merge(BR[['pc11_state_id','pc11_district_id','dist_code']], on=['pc11_state_id','pc11_district_id'])
        .merge(W[['pc11_state_id','pc11_district_id','pop']], on=['pc11_state_id','pc11_district_id']))
NF = (nf.groupby('dist_code')
        .apply(lambda g: pd.Series({
            'nonfarm_sh': np.average(g.nonfarm_sh, weights=g['pop']),
            'hhind_sh'  : np.average(g.hhind_sh,   weights=g['pop']),
            'ec_emp_pc' : np.average(g.ec_emp_pc.fillna(0), weights=g['pop'])}))
        .reset_index())

# ---- rainfall shock ----
r = pd.read_parquet(f'{BASE}/rainfall_shock.parquet'); r = r[r['Dist Code'] < 800]
r = r.rename(columns={'Dist Code':'dist_code', 'Year':'year'}) # Renamed 'Year' to 'year'
P = L.merge(r[['dist_code','year','rain_z','rain_dev']], # Select 'year' now that it exists
            on=['dist_code','year'], # Merge directly on 'dist_code' and 'year'
            how='inner').merge(NF, on='dist_code')
P['z_neg'] = np.minimum(P.rain_z, 0)
P['drought'] = (P.rain_dev < -0.20).astype(float)
print('H4 panel:', P.shape, '|', P.dist_code.nunique(), 'districts x', P.year.nunique(), 'years')

def dm(df, cols, it=12):
    X = df[cols].astype(float).copy()
    for _ in range(it):
        X = X - X.groupby(df['dist_code'].values).transform('mean')
        X = X - X.groupby(df['year'].values).transform('mean')
    return X

for bufvar in ['nonfarm_sh','hhind_sh','ec_emp_pc']:
    for shock in ['z_neg','drought']:
        Q = P.dropna(subset=['ln_light',shock,bufvar]).copy()
        Q['b_c']   = Q[bufvar] - Q[bufvar].mean()
        Q['inter'] = Q[shock] * Q.b_c
        D = dm(Q, ['ln_light', shock, 'inter'])
        m = sm.OLS(D.ln_light.values, D[[shock,'inter']].values).fit(
              cov_type='cluster', cov_kwds={'groups': Q.dist_code.values})
        lo, hi = Q[bufvar].quantile(.25), Q[bufvar].quantile(.75)
        print(f'\n{bufvar:11s} x {shock:8s}  N {len(Q)}')
        print(f'   shock      {m.params[0]:+.4f} (t {m.tvalues[0]:.1f})')
        print(f'   x buffer   {m.params[1]:+.4f} (t {m.tvalues[1]:.1f})   <- NEGATIVE = buffer works')
        print(f'   effect at p25 buffer ({lo:.3f}): {m.params[0]+m.params[1]*(lo-Q[bufvar].mean()):+.4f}')
        print(f'   effect at p75 buffer ({hi:.3f}): {m.params[0]+m.params[1]*(hi-Q[bufvar].mean()):+.4f}')

P.to_parquet(f'{BASE}/h4_panel.parquet', index=False)
print('\nsaved h4_panel.parquet')


/tmp/ipykernel_2077/2185091038.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({'light': g.wl.sum()/g['pop'].sum()}))


light panel: (6096, 4) 1994 - 2013 | 305 districts


/tmp/ipykernel_2077/2185091038.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


H4 panel: (5111, 11) | 275 districts x 20 years

nonfarm_sh  x z_neg     N 5022
   shock      +0.0059 (t 0.6)
   x buffer   +0.0857 (t 2.0)   <- NEGATIVE = buffer works
   effect at p25 buffer (0.221): -0.0047
   effect at p75 buffer (0.434): +0.0135

nonfarm_sh  x drought   N 5111
   shock      -0.0040 (t -0.4)
   x buffer   -0.0418 (t -0.8)   <- NEGATIVE = buffer works
   effect at p25 buffer (0.221): +0.0013
   effect at p75 buffer (0.437): -0.0078

hhind_sh    x z_neg     N 5022
   shock      +0.0038 (t 0.4)
   x buffer   +0.9069 (t 1.8)   <- NEGATIVE = buffer works
   effect at p25 buffer (0.013): -0.0055
   effect at p75 buffer (0.027): +0.0076

hhind_sh    x drought   N 5111
   shock      -0.0046 (t -0.5)
   x buffer   -2.3915 (t -3.2)   <- NEGATIVE = buffer works
   effect at p25 buffer (0.013): +0.0201
   effect at p75 buffer (0.027): -0.0144

ec_emp_pc   x z_neg     N 5022
   shock      +0.0075 (t 0.7)
   x buffer   +0.3992 (t 0.7)   <- NEGATIVE = buffer works
   effect at p2

In [ ]:
# SHRUG CELL 6 — is there a first stage at all? test each link in the chain
import pandas as pd, numpy as np, statsmodels.api as sm
BASE = '/content/drive/My Drive/Data/Elevar'
P = pd.read_parquet(f'{BASE}/h4_panel.parquet')
crops = pd.read_parquet(f'{BASE}/crops_clean.parquet').rename(columns={'Dist Code':'dist_code','Year':'year'})
P = P.merge(crops[['dist_code','year','ln_ag_output']], on=['dist_code','year'], how='left')

def dm(df, cols, it=12):
    X = df[cols].astype(float).copy()
    for _ in range(it):
        X = X - X.groupby(df['dist_code'].values).transform('mean')
        X = X - X.groupby(df['year'].values).transform('mean')
    return X
def fe(df, y, x):
    Q = df.dropna(subset=[y]+x); D = dm(Q, [y]+x)
    m = sm.OLS(D[y].values, D[x].values).fit(cov_type='cluster', cov_kwds={'groups':Q.dist_code.values})
    return m, len(Q)

print('LINK 1  rain -> ag output   (known to work; confirms the sample)')
m,n = fe(P,'ln_ag_output',['z_neg']); print(f'   z_neg {m.params[0]:+.4f} (t {m.tvalues[0]:.1f}) N {n}')

print('\nLINK 2  rain -> light')
for s in ['z_neg','drought','rain_z']:
    m,n = fe(P,'ln_light',[s]); print(f'   {s:8s} {m.params[0]:+.4f} (t {m.tvalues[0]:.1f}) N {n}')

print('\nLINK 3  ag output -> light   (does local farm income move lights AT ALL?)')
m,n = fe(P,'ln_light',['ln_ag_output']); print(f'   ln_ag_output {m.params[0]:+.4f} (t {m.tvalues[0]:.1f}) N {n}')

print('\nLINK 4  lagged: does light respond a year later?')
P = P.sort_values(['dist_code','year'])
P['z_neg_l1'] = P.groupby('dist_code').z_neg.shift(1)
P['z_neg_l2'] = P.groupby('dist_code').z_neg.shift(2)
m,n = fe(P,'ln_light',['z_neg','z_neg_l1','z_neg_l2'])
for i,lab in enumerate(['t','t-1','t-2']): print(f'   {lab:4s} {m.params[i]:+.4f} (t {m.tvalues[i]:.1f})')

print('\nDIAGNOSTIC  how much of light is trend vs shock?')
for lab, sub in [('all', P), ('1994-2003', P[P.year<=2003]), ('2004-2013', P[P.year>2003])]:
    m,n = fe(sub,'ln_light',['z_neg'])
    print(f'   {lab:10s} z_neg {m.params[0]:+.4f} (t {m.tvalues[0]:.1f})  mean ln_light {sub.ln_light.mean():.2f}')

LINK 1  rain -> ag output   (known to work; confirms the sample)
   z_neg +0.1350 (t 10.0) N 5013

LINK 2  rain -> light
   z_neg    +0.0056 (t 0.6) N 5022
   drought  -0.0036 (t -0.4) N 5111
   rain_z   -0.0031 (t -0.6) N 5022

LINK 3  ag output -> light   (does local farm income move lights AT ALL?)
   ln_ag_output -0.0385 (t -1.1) N 5093

LINK 4  lagged: does light respond a year later?
   t    -0.0085 (t -1.3)
   t-1  +0.0197 (t 3.2)
   t-2  -0.0094 (t -1.3)

DIAGNOSTIC  how much of light is trend vs shock?
   all        z_neg +0.0056 (t 0.6)  mean ln_light 1.42
   1994-2003  z_neg +0.0046 (t 0.4)  mean ln_light 1.28
   2004-2013  z_neg +0.0008 (t 0.1)  mean ln_light 1.55


In [ ]:
# SHOT 3A — build SHRUG district features, aggregate to ICRISAT parents
import pandas as pd, numpy as np, re
BASE = '/content/drive/My Drive/Data/Elevar'
BR = pd.read_csv(f'{BASE}/pc11_to_icrisat_bridge.csv')
W  = S['distwt'].rename(columns={'dist_pc11_pca_tot_p':'pop','dist_pc11_land_area':'area'})
PKEY = ['pc11_state_id','pc11_district_id']

def grab(df, pats, label):
    """find columns matching any regex; report hits"""
    hits = [c for c in df.columns if any(re.search(p, c, re.I) for p in pats)]
    print(f'  {label:22s} -> {hits[:6]}{" ..." if len(hits)>6 else ""}')
    return hits

F = W[PKEY+['pop','area']].copy()

# ---- terrain (exogenous) ----
F = F.merge(S['elev'][PKEY+['elevation_mean','elevation_std']], on=PKEY, how='left')
F = F.merge(S['tri'][PKEY+['tri_mean']], on=PKEY, how='left')

# ---- forest + pollution, recent mean ----
for tag, df, col in [('vcf','vcf','vcf_mean'), ('pm25','pm25','pm25_mean')]:
    d = S[df]; d = d[d.year.between(2005,2013)]
    F = F.merge(d.groupby(PKEY)[col].mean().rename(tag).reset_index(), on=PKEY, how='left')

# ---- census 2011 worker structure + demographics ----
p = S['pca11']
F = F.merge(p[PKEY+['pc11_pca_tot_p','pc11_pca_tot_work_p','pc11_pca_main_cl_p','pc11_pca_main_al_p',
                    'pc11_pca_main_hh_p','pc11_pca_main_ot_p','pc11_pca_p_lit','pc11_pca_p_st',
                    'pc11_pca_p_sc','pc11_pca_no_hh']], on=PKEY, how='left')
tw = F.pc11_pca_tot_work_p
F['nonfarm_sh'] = (F.pc11_pca_main_hh_p + F.pc11_pca_main_ot_p)/tw
F['hhind_sh']   =  F.pc11_pca_main_hh_p/tw
F['cultiv_sh']  =  F.pc11_pca_main_cl_p/tw
F['aglab_sh']   =  F.pc11_pca_main_al_p/tw
F['wfpr']       =  tw/F.pc11_pca_tot_p
F['literacy2']  =  F.pc11_pca_p_lit/F.pc11_pca_tot_p
F['st_sh']      =  F.pc11_pca_p_st/F.pc11_pca_tot_p
F['sc_sh']      =  F.pc11_pca_p_sc/F.pc11_pca_tot_p
F['hh_size']    =  F.pc11_pca_tot_p/F.pc11_pca_no_hh

# ---- economic census 2013: employment + sector mix ----
print('\nEC13 column search:')
e = S['ec13']
sec = grab(e, [r'ec13_emp_manuf', r'ec13_emp_services', r'ec13_emp_shric'], 'sector cols')
keep = PKEY+['ec13_emp_all','ec13_count_all','ec13_emp_hired','ec13_emp_unhired']+ \
       [c for c in ['ec13_emp_manuf','ec13_emp_services'] if c in e.columns]
F = F.merge(e[[c for c in keep if c in e.columns]], on=PKEY, how='left')
F['ec_emp_pc']    = F.ec13_emp_all/F.pc11_pca_tot_p
F['ec_firm_pc']   = F.ec13_count_all/F.pc11_pca_tot_p
F['ec_firm_size'] = F.ec13_emp_all/F.ec13_count_all
if 'ec13_emp_unhired' in F: F['ec_ownaccount_sh'] = F.ec13_emp_unhired/F.ec13_emp_all
if 'ec13_emp_manuf' in F:   F['ec_manuf_sh']      = F.ec13_emp_manuf/F.ec13_emp_all

# ---- SECC rural: land, irrigation, jobs ----
print('\nSECC rural column search:')
sr = S['secc_rural']
grab(sr, [r'land_own', r'cultiv', r'irr', r'acre', r'jobs'], 'land/irr/jobs')
for c in ['land_own_share','nco2d_cultiv_share','unirr_land_acre_sum','two_crop_acre_sum',
          'other_irr_acre_sum','jobs_1d_p','secc_hh']:
    if c in sr.columns: F = F.merge(sr[PKEY+[c]], on=PKEY, how='left')
if 'unirr_land_acre_sum' in F and 'other_irr_acre_sum' in F:
    tot = F.unirr_land_acre_sum + F.other_irr_acre_sum + F.get('two_crop_acre_sum',0)
    F['secc_unirr_sh'] = F.unirr_land_acre_sum/tot.replace(0,np.nan)
if 'jobs_1d_p' in F and 'secc_hh' in F: F['secc_jobs_pc'] = F.jobs_1d_p/F.secc_hh

# ---- consumption level + poverty (structural wealth, not the outcome) ----
F = F.merge(S['secc_cons'][PKEY+['secc_cons_pc_rural','secc_pov_rate_rural']], on=PKEY, how='left')

# ---- market access from town directory ----
print('\nTD11 column search:')
td = S['td11']; grab(td, [r'city_dist', r'rail_dist', r'city5_dist'], 'distance cols')
for c in ['pc11_td_city_dist','pc11_td_rail_dist','pc11_td_city5_dist']:
    if c in td.columns:
        F = F.merge(td.groupby(PKEY)[c].mean().reset_index(), on=PKEY, how='left')

# ---- village directory: credit + infrastructure ----
print('\nVD11 column search:')
vd = S['vd11']
cred = grab(vd, [r'comm_bank', r'coop_bank', r'ag_credit', r'credit_soc', r'_bank'], 'credit')
road = grab(vd, [r'pucca_road', r'tar_road', r'all_weather', r'_road'], 'roads')
powr = grab(vd, [r'power_supply', r'power_agri', r'electri'], 'power')
for c in (cred[:3]+road[:3]+powr[:3]):
    if vd[c].dtype.kind in 'if':
        F = F.merge(vd[PKEY+[c]].rename(columns={c:'vd_'+c[-28:]}), on=PKEY, how='left')

# ---- aggregate to ICRISAT parents, population-weighted ----
F = F.merge(BR[PKEY+['dist_code']], on=PKEY, how='inner')
NUM = [c for c in F.columns if c not in PKEY+['dist_code'] and F[c].dtype.kind in 'if']
def wavg(g, c):
    v, w = g[c], g['pop']
    m = v.notna()
    return np.average(v[m], weights=w[m]) if m.any() and w[m].sum()>0 else np.nan
G = F.groupby('dist_code').apply(lambda g: pd.Series({c: wavg(g,c) for c in NUM}), include_groups=False)

print(f'\nSHRUG feature block: {G.shape[0]} ICRISAT parents x {G.shape[1]} features')
cov = (G.notna().mean()*100).round(1).sort_values()
print('\nlowest coverage:'); print(cov.head(10).to_string())
G.reset_index().to_parquet(f'{BASE}/shrug_features.parquet', index=False)
print('\nsaved shrug_features.parquet')


EC13 column search:
  sector cols            -> ['ec13_emp_manuf', 'ec13_emp_services', 'ec13_emp_shric_1', 'ec13_emp_shric_2', 'ec13_emp_shric_3', 'ec13_emp_shric_4'] ...

SECC rural column search:
  land/irr/jobs          -> ['land_own_share', 'nco2d_cultiv_share', 'unirr_land_acre_sum', 'two_crop_acre_sum', 'other_irr_acre_sum', 'jobs_1d_p'] ...

TD11 column search:
  distance cols          -> ['pc11_td_city_dist', 'pc11_td_city5_dist', 'pc11_td_rail_dist']

VD11 column search:
  credit                 -> ['pc11_vd_comm_bank', 'pc11_vd_coop_bank']
  roads                  -> []
  power                  -> []

SHRUG feature block: 305 ICRISAT parents x 53 features

lowest coverage:
nco2d_cultiv_share     80.7
two_crop_acre_sum      96.4
jobs_1d_p              96.4
other_irr_acre_sum     96.4
secc_unirr_sh          96.4
secc_pov_rate_rural    96.4
secc_jobs_pc           96.4
secc_hh                96.4
land_own_share         96.4
unirr_land_acre_sum    96.4

saved shrug_features.parq

In [ ]:
# SHOT 3B — refit exposure model on ICRISAT ∪ SHRUG features
import pandas as pd, numpy as np, shap
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
BASE = '/content/drive/My Drive/Data/Elevar'

IC = pd.read_parquet(f'{BASE}/features.parquet').set_index('Dist Code')
SH = pd.read_parquet(f'{BASE}/shrug_features.parquet').set_index('dist_code')
D  = IC.join(SH, how='left', rsuffix='_sh')
D  = D.dropna(subset=['exposure'])
y  = D.exposure.astype(float).values
w  = D['w'].astype(float).values if 'w' in D else np.ones(len(D))
cv = KFold(5, shuffle=True, random_state=7)
def r2(yt,yp): return 1-((yt-yp)**2).sum()/((yt-yt.mean())**2).sum()

BASE_F = ['coarse_cereal_share','oilseeds_share','cash_crop_share','irr_share','canal_share',
          'tubewell_share','kharif_normal','rain_cv','crop_intensity','marginal_share',
          'pop_density','banks_per_lakh','milk_kg_pc']
NEW_F  = ['elevation_mean','elevation_std','tri_mean','vcf','pm25',
          'nonfarm_sh','hhind_sh','cultiv_sh','aglab_sh','wfpr','literacy2','st_sh','sc_sh','hh_size',
          'ec_emp_pc','ec_firm_pc','ec_firm_size','ec_ownaccount_sh','ec_manuf_sh',
          'land_own_share','secc_unirr_sh','secc_jobs_pc','secc_cons_pc_rural','secc_pov_rate_rural',
          'pc11_td_city_dist','pc11_td_rail_dist','pc11_vd_comm_bank','pc11_vd_coop_bank']
BASE_F = [c for c in BASE_F if c in D.columns]
NEW_F  = [c for c in NEW_F  if c in D.columns and D[c].notna().mean() > 0.80]
print(f'N {len(D)} | base {len(BASE_F)} | new {len(NEW_F)}')

def fit(feats, tag, verbose=False):
    X = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(D[feats].astype(float)),
                     columns=feats, index=D.index)
    g = HistGradientBoostingRegressor(max_depth=3, max_iter=400, learning_rate=0.05,
                                      min_samples_leaf=15, l2_regularization=1.0, random_state=7)
    oof = cross_val_predict(g, X, y, cv=cv, params={'sample_weight': w})
    lin = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3,3,25)))
    oofl = cross_val_predict(lin, X, y, cv=cv)
    print(f'{tag:24s} gbm {r2(y,oof):.3f} | ridge {r2(y,oofl):.3f} | rank {pd.Series(y).corr(pd.Series(oof),method="spearman"):.3f}')
    g.fit(X, y, sample_weight=w)
    return g, X, r2(y,oof)

print('\n--- OUT-OF-FOLD R2  (ceiling = reliability 0.92) ---')
_,_,r_base = fit(BASE_F,               'ICRISAT only')
_,_,r_new  = fit(NEW_F,                'SHRUG only')
gU,XU,r_un = fit(BASE_F+NEW_F,         'UNION')

# --- backward selection on the union: drop until oof stops improving ---
feats = BASE_F + NEW_F
best = r_un
print(f'\n--- pruning from {len(feats)} features (start {best:.3f}) ---')
while len(feats) > 8:
    g = HistGradientBoostingRegressor(max_depth=3,max_iter=400,learning_rate=0.05,
                                      min_samples_leaf=15,l2_regularization=1.0,random_state=7)
    X = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(D[feats].astype(float)),
                     columns=feats, index=D.index)
    g.fit(X,y,sample_weight=w)
    imp = pd.DataFrame(shap.TreeExplainer(g).shap_values(X),columns=feats).abs().mean()
    drop = imp.idxmin()
    trial = [f for f in feats if f != drop]
    Xt = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(D[trial].astype(float)),
                      columns=trial, index=D.index)
    oof = cross_val_predict(HistGradientBoostingRegressor(max_depth=3,max_iter=400,learning_rate=0.05,
                            min_samples_leaf=15,l2_regularization=1.0,random_state=7),
                            Xt,y,cv=cv,params={'sample_weight':w})
    rt = r2(y,oof)
    if rt >= best - 0.002:
        feats, best = trial, max(best, rt)
        print(f'  drop {drop:24s} -> {rt:.3f}  ({len(feats)} left)')
    else:
        print(f'  stop: dropping {drop} would cost {best-rt:.3f}')
        break

print(f'\nFINAL: {len(feats)} features, oof R2 {best:.3f}  (was {r_base:.3f} ICRISAT-only)')
X = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(D[feats].astype(float)),
                 columns=feats, index=D.index)
gF = HistGradientBoostingRegressor(max_depth=3,max_iter=400,learning_rate=0.05,
                                   min_samples_leaf=15,l2_regularization=1.0,random_state=7)
gF.fit(X,y,sample_weight=w)
SV = pd.DataFrame(shap.TreeExplainer(gF).shap_values(X), columns=feats, index=D.index)
print('\nmean |SHAP|:'); print(SV.abs().mean().sort_values(ascending=False).round(4).to_string())
print('\nlead driver spread:'); print(SV.abs().idxmax(axis=1).value_counts().head(8).to_string())
print('\ncorrelation with exposure, new features only:')
print(D[[f for f in feats if f in NEW_F]+['exposure']].corr()['exposure'].drop('exposure').sort_values().round(3).to_string())

SV.join(D[['Dist Name','State Name']]).reset_index().to_parquet(f'{BASE}/shap_union.parquet', index=False)
D.reset_index().to_parquet(f'{BASE}/features_union.parquet', index=False)
print('\nsaved shap_union.parquet, features_union.parquet')

N 268 | base 13 | new 26

--- OUT-OF-FOLD R2  (ceiling = reliability 0.92) ---
ICRISAT only             gbm 0.495 | ridge 0.407 | rank 0.649
SHRUG only               gbm 0.429 | ridge 0.369 | rank 0.667
UNION                    gbm 0.501 | ridge 0.491 | rank 0.666

--- pruning from 39 features (start 0.501) ---
  drop pc11_td_rail_dist        -> 0.499  (38 left)
  stop: dropping pc11_td_city_dist would cost 0.005

FINAL: 38 features, oof R2 0.501  (was 0.495 ICRISAT-only)

mean |SHAP|:
pop_density            0.0469
rain_cv                0.0247
secc_unirr_sh          0.0224
oilseeds_share         0.0206
coarse_cereal_share    0.0205
kharif_normal          0.0194
tubewell_share         0.0169
vcf                    0.0156
marginal_share         0.0152
sc_sh                  0.0152
wfpr                   0.0125
literacy2              0.0123
secc_jobs_pc           0.0104
elevation_mean         0.0101
irr_share              0.0101
pm25                   0.0095
st_sh                  0.0084